In [ ]:
!pip install nipype nilearn nibabel

In [ ]:
import os

data_dir = 'data/ds000051'

print("Data directory contents:", os.listdir(data_dir))
if 'sub-01' in os.listdir(data_dir):
    print("Sub-01 directory contents:", os.listdir(os.path.join(data_dir, 'sub-01')))
    if 'anat' in os.listdir(os.path.join(data_dir, 'sub-01')):
        print("Sub-01 anat directory contents:", os.listdir(os.path.join(data_dir, 'sub-01', 'anat')))

In [ ]:
import os
from nipype import Node, Workflow
from nipype.interfaces import fsl
from nipype.interfaces.io import SelectFiles, DataSink
from nipype.algorithms.confounds import TSNR
from nipype.interfaces.fsl import ExtractROI, FLIRT, ImageMeants
from nipype.interfaces.utility import Function

# Paths
data_dir = 'data/ds000051'
output_dir = 'data/output'
mni_template = 'data/ROI/MNI152_T1_1mm_brain.nii.gz'

# Create SelectFiles node
templates = {
    'func': os.path.join(data_dir, 'sub-01/func/sub-01_task-abstractconcretejudgment_run-01_bold.nii.gz'),
    'anat': os.path.join(data_dir, 'sub-01/anat/sub-01_run-01_T1w.nii.gz'),
}
selectfiles = Node(SelectFiles(templates), name='selectfiles')
selectfiles.inputs.base_directory = data_dir
selectfiles.inputs.sort_filelist = True

# TSNR node
compute_tsnr = Node(TSNR(), name='compute_tsnr')

# Extract ROI node
extract_rois = Node(ExtractROI(t_min=0, t_size=1), name='extract_rois')

# Atlas-based ROI extraction
atlas_img_path = mni_template
if not os.path.exists(atlas_img_path):
    raise FileNotFoundError(f"Atlas image not found at {atlas_img_path}")

# Define ROI extraction function
def extract_mean_timeseries(in_file, atlas_img):
    imgmeants = ImageMeants(in_file=in_file, mask=atlas_img)
    return imgmeants.run().outputs.out_file

extract_mean_roi = Node(Function(input_names=['in_file', 'atlas_img'],
                                 output_names=['out_file'],
                                 function=extract_mean_timeseries),
                        name='extract_mean_roi')
extract_mean_roi.inputs.atlas_img = atlas_img_path

# FLIRT registration to MNI space for anatomical and functional data
reg_anat = Node(FLIRT(reference=mni_template), name='reg_anat')
reg_func = Node(FLIRT(reference=mni_template), name='reg_func')

# Compute connectivity function
def compute_correlation(in_file):
    import nibabel as nib
    import numpy as np
    from scipy.stats import pearsonr

    img = nib.load(in_file)
    data = img.get_fdata()
    n_voxels = np.prod(data.shape[:-1])
    data_2d = np.reshape(data, (n_voxels, data.shape[-1])).T
    corr_matrix = np.zeros((n_voxels, n_voxels))

    for i in range(n_voxels):
        for j in range(i, n_voxels):
            corr_matrix[i, j] = pearsonr(data_2d[i], data_2d[j])[0]
            corr_matrix[j, i] = corr_matrix[i, j]

    return corr_matrix

compute_conn = Node(Function(input_names=['in_file'],
                             output_names=['correlation_matrix'],
                             function=compute_correlation),
                    name='compute_conn')

# DataSink node
datasink = Node(DataSink(base_directory=output_dir), name="datasink")
datasink.inputs.container = "results"

# Workflow definition
wf = Workflow(name='functional_connectivity_analysis', base_dir=output_dir)

# Connect the nodes
wf.connect([
    (selectfiles, compute_tsnr, [('func', 'in_file')]),
    (compute_tsnr, extract_rois, [('detrended_file', 'in_file')]),
    (extract_rois, reg_func, [('roi_file', 'in_file')]),
    (selectfiles, reg_func, [('func', 'reference')]),
    (selectfiles, reg_anat, [('anat', 'in_file')]),
    (reg_anat, extract_mean_roi, [('out_file', 'in_file')]),
    (extract_mean_roi, compute_conn, [('out_file', 'in_file')]),
    (compute_conn, datasink, [('correlation_matrix', '@connectivity_matrix')]),
])

# Run the workflow
try:
    wf.run('MultiProc', plugin_args={'n_procs': 4})
except Exception as e:
    print(f"Error occurred: {e}")

In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the output directory
output_dir = 'results'

# Define the directories for lesions and connectivity
connectivity_dir = os.path.join(output_dir, 'connectivity')

# Read and visualize functional connectivity matrices
def visualize_connectivity(subject_id):
    conn_file = os.path.join(connectivity_dir, f'correlation_matrix_sub-{subject_id}.npy')
    if os.path.exists(conn_file):
        correlation_matrix = np.load(conn_file)

        # Display the connectivity matrix
        plt.figure(figsize=(10, 10))
        sns.heatmap(correlation_matrix, square=True, cmap='coolwarm', vmin=-1, vmax=1)
        plt.title(f'Functional Connectivity Matrix for Subject {subject_id}')
        plt.show()
    else:
        print(f"Connectivity matrix for subject {subject_id} not found.")

# Example usage
subject_list = ['01', '02']
for subject_id in subject_list:
    print(f"Visualizing outputs for subject {subject_id}")
    visualize_connectivity(subject_id)